# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a practical walkthrough for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library.

### Dataset Source
This dataset describes a mass spectrometry analysis of protein abundance, modifications, and sequences in 
human (Homo sapiens) samples. It has multiple fields related to protein characteristics, measured and calculated columns, and is described using a [Croissant schema](https://mlcommons.org/croissant/).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Metadata URL: {croissant_url}")

## 2. Data Overview
Review available record sets, their `@id`s, and the structure of fields for each record set. This will guide us in selecting the right data to analyze.

We will enumerate all record sets, along with their fields and columns, using the dataset metadata.

In [ ]:
# Show a summary of all record sets with their @ids, fields, and columns
record_set_ids = []

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        print(f"Record Set: {rs.name} (id: {rs['@id']})")
        record_set_ids.append(rs['@id'])
        if hasattr(rs, 'field') and rs.field:
            print("  Fields:")
            for f in rs.field:
                fname = getattr(f, 'name', None)
                f_id = f['@id'] if '@id' in f else ''
                print(f"    - {fname} (@id: {f_id})")
                if hasattr(f, 'column') and f.column:
                    for c in f.column:
                        cname = getattr(c, 'name', None)
                        cid = c['@id'] if '@id' in c else ''
                        print(f"      Column: {cname} (@id: {cid})")
        print()
else:
    print("No record sets found in the metadata. This dataset may not expose records via Croissant's recordSet. You may need to check the Croissant schema or use another method to explore files/distributions.")

## 3. Data Extraction
If the dataset contains record sets, load data from those record sets using their `@id` fields. 
If none are present, demonstrate how to load distribution files directly with `mlcroissant`.

> *For this dataset, we will check for record sets and fallback to distribution files if none are found.*

In [ ]:
import warnings

# Try to extract data from record sets, or fallback to file-based loading
dataframes = {}

if record_set_ids:
    # Extract all record sets
    for rs_id in record_set_ids:
        print(f"Loading records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Record set '{rs_id}' - columns: {df.columns.tolist()}")
            display(df.head())
        else:
            warnings.warn(f"No records could be loaded for record set {rs_id}.")
    # For demonstration, pick the first record set for further analysis
    recset_for_analysis = record_set_ids[0]
else:
    print("No record sets found; fallback to distribution files.")
    # Fallback: Try loading first distribution file
    if hasattr(metadata, 'distribution') and metadata.distribution:
        main_file_url = None
        from urllib.parse import urlparse

        dist = metadata.distribution[0]
        if hasattr(dist, 'contentUrl'):
            if isinstance(dist.contentUrl, list):
                main_file_url = dist.contentUrl[0]
            else:
                main_file_url = dist.contentUrl
            print(f"Main data file URL: {main_file_url}")
            # Try to read with pandas if CSV/TSV
            if main_file_url.lower().endswith('.csv'):
                df = pd.read_csv(main_file_url)
            elif main_file_url.lower().endswith('.tsv'):
                df = pd.read_csv(main_file_url, sep='\t')
            else:
                print("File is not a CSV or TSV; manual inspection needed.")
                df = None
            if df is not None:
                dataframes['main'] = df
                print(f"Loaded file columns: {df.columns.tolist()}")
                display(df.head())
        else:
            print("Distribution file does not have a contentUrl.")
    else:
        print("No distribution files found in Croissant metadata.")
    # For further code, use the main dataframe
    recset_for_analysis = list(dataframes.keys())[0] if dataframes else None


## 4. Exploratory Data Analysis (EDA)
Explore and process the dataframe: filter by a numeric column, normalize it, group by a category if present.

We will select a numeric column (e.g., 'MW_kDa' or another available) and a suitable grouping column if present.

In [ ]:
if recset_for_analysis and recset_for_analysis in dataframes:
    df = dataframes[recset_for_analysis]
    # Try to find a numeric field: common protein datasets have MW, Peptides, Coverage, etc.
    import numpy as np
    
    # Try to auto-select a numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not numeric_candidates:
        # Try to convert likely columns
        possible_numeric = ['MW_kDa','Peptides','Coverage','Score','Abundance','MW']
        for col in possible_numeric:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()  # Use mean as threshold for demo
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Now find a likely grouping field (e.g., 'Category', 'Sample', 'Protein Class', etc.)
        group_candidates = [c for c in df.columns if c.lower() in ['sample','category','group']]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found for aggregation.")
    else:
        print("Could not find a numeric field suitable for analysis.")
else:
    print("No structured dataframe loaded for EDA.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and show a bar plot if grouping is possible.

*This requires matplotlib or seaborn.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if recset_for_analysis and recset_for_analysis in dataframes and 'numeric_field' in locals():
    df = dataframes[recset_for_analysis]
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Bar plot by group if available
    if 'group_field' in locals():
        group_means = df.groupby(group_field)[numeric_field].mean()
        group_means = group_means.dropna()
        group_means.sort_values(ascending=False).plot(kind='bar', figsize=(9, 4))
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No dataframe/field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the FAIR² mass spectrometry dataset using the [mlcroissant](https://mlcommons.github.io/croissant/python/) library. We explored its schema, structured records, conducted basic filtering and normalization, and produced visualizations of key measurements.

This workflow can be extended to other FAIR-style datasets declared with Croissant schemas for robust and reproducible data science.